# Where pandas Stops

pandas is great, but it has one big limit: **all the data must fit in the memory (RAM) of one machine**. And in memory the data takes much more space than in the file on disk.

In this notebook we load one year of New York taxi trips with pandas, month by month, and watch the memory grow. Run it on **Google Colab**: the free version gives you about 12 GB of RAM.

**The data:** every taxi trip in New York City, published by the [NYC Taxi and Limousine Commission](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page) as one Parquet file per month. In 2024 there were about **3.4 million trips per month**, **41 million** in the whole year. Each trip has 19 columns: pickup and drop-off time, distance, fare, tip, payment type, and more (see the [data dictionary](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf)).

## 0. How much memory do we have?

In [ ]:
import os
import time
import urllib.request

import pandas as pd
import psutil

GB = 1024 ** 3

def show_memory(label=""):
    """Memory of this notebook (its Python process) and of the whole machine."""
    notebook = psutil.Process().memory_info().rss
    m = psutil.virtual_memory()
    print(f"{label}this notebook: {notebook / GB:.1f} GB | machine: {m.used / GB:.1f} GB used, {m.available / GB:.1f} GB free of {m.total / GB:.1f} GB")

show_memory()

In [ ]:
BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"
DATA_DIR = "taxi_data"
os.makedirs(DATA_DIR, exist_ok=True)

def download_month(month):
    """Download one month of 2024 (if not already done) and return the local path."""
    name = f"yellow_tripdata_2024-{month:02d}.parquet"
    path = os.path.join(DATA_DIR, name)
    if not os.path.exists(path):
        urllib.request.urlretrieve(f"{BASE_URL}/{name}", path)
    return path

## 1. One month: no problem

We load January 2024 and compare the size of the file with the memory the DataFrame uses.

In [ ]:
path = download_month(1)
january = pd.read_parquet(path)

file_mb = os.path.getsize(path) / 1024 ** 2
memory_mb = january.memory_usage(deep=True).sum() / 1024 ** 2

print(f"rows: {len(january):,}")
print(f"file on disk: {file_mb:.0f} MB | in pandas: {memory_mb:.0f} MB (about {memory_mb / file_mb:.0f} times bigger)")
show_memory()

### Why is it so much bigger in memory?

Three reasons:

1. **pandas stores every value at full size.** Every number takes 8 bytes, whatever its value: a column of 3 million numbers always takes about 23 MB. Parquet notices when a column has few different values and stores each value only once, plus small codes that point to it (*dictionary encoding*). For example `VendorID` has only 3 different values: 11.3 MB in pandas, 0.3 MB on disk.
2. **Text is very expensive in pandas.** The column `store_and_fwd_flag` only contains "Y" or "N", but pandas stores each one as a separate Python object of about 50 bytes. This single column takes about **140 MB**, more than a quarter of the total, while on disk it takes almost nothing.
3. **Parquet is compressed.** Because Parquet is a columnar format, similar values are stored next to each other and compress very well. Columns with many different values compress less: the pickup times (1.5 million different values) only go from 23 MB to 12 MB.

**The lesson:** the size of a file tells you little about the memory you need. With Parquet files, expect pandas to need 5 to 10 times the size of the file, or more if there is text.

A question on one month is fast: the average tip by payment type.

Payment types: 0 = flex fare, 1 = credit card, 2 = cash, 3 = no charge, 4 = dispute, 5 = unknown, 6 = voided trip. Tips paid in cash are not recorded, so for cash they look like 0.

In [ ]:
start = time.perf_counter()
result = january.groupby("payment_type")["tip_amount"].mean().round(2)
print(f"query time: {time.perf_counter() - start:.2f} s")
result

## 2. The whole year: memory grows month by month

Now we load the 12 months one by one and keep them all in memory, as you would to analyze the whole year. Watch the memory grow, and the time it takes.

In [ ]:
del january

months = []
rows, used, loading_time = 0, 0, 0
for month in range(1, 13):
    path = download_month(month)  # the download is not counted in the time

    start = time.perf_counter()
    month_df = pd.read_parquet(path)
    loading_time += time.perf_counter() - start

    months.append(month_df)
    rows += len(month_df)
    used += month_df.memory_usage(deep=True).sum() / GB
    print(f"month {month:2d}: {rows:>12,} rows | loading time: {loading_time:4.1f} s | data in pandas: {used:4.1f} GB | ", end="")
    show_memory()

To analyze the whole year as one table, pandas has to **merge** the 12 months with `pd.concat`. This creates a **new copy** of all the data, so for a moment we need the memory twice.

The next cell checks if there is enough free memory before trying. (Without this check, Colab would simply crash and restart, losing everything.)

In [ ]:
needed = sum(m.memory_usage(deep=True).sum() for m in months)
free = psutil.virtual_memory().available

print(f"memory needed for the merge: {needed / GB:.1f} GB | free: {free / GB:.1f} GB")
if needed > free * 0.9:
    print("Not enough memory: pandas cannot hold the whole year as one table on this machine.")
else:
    year = pd.concat(months, ignore_index=True)
    print(f"Merged: {len(year):,} rows. This machine has a lot of memory: try 2 or 3 years and see what happens.")
show_memory()

## What did we learn?

- One month (3 million trips, a 48 MB file) already takes about **0.5 GB** in pandas.
- One year (41 million trips, about 650 MB of files) takes about **7 GB**, and twice that to merge it.
- The memory needed grows with the data. Ten years of taxi trips, or the logs of a big website, will **never** fit in one machine, however big.

pandas cannot go beyond the memory of one computer. **Spark** can: it splits the data across many machines, and it reads from disk only what it needs. In the next notebook we ask the same questions on the same year, with Spark on Databricks.

In [ ]:
# Free the memory
del months
show_memory()